RAG Agent: Cited Sources + RAG Triad Scoring + SKILL.md

No API key, no quotas. Everything runs inside Colab on open models:
- Embeddings: `all-MiniLM-L6-v2` (sentence-transformers)
- LLM (agent + judge): `Qwen2.5-3B-Instruct`
- Vector store: Chroma

**Before running:** Runtime -> Change runtime type -> **T4 GPU** (free). First run downloads the models (a few minutes).

**Outcome this notebook produces**
1. An agent that answers domain questions **with cited sources**
2. Scores on the **RAG triad**: context relevance, groundedness, answer relevance
3. A **SKILL.md** to hand to a peer who hasn't seen the agent

In [1]:
!pip -q install chromadb sentence-transformers transformers accelerate pandas

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 81.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 28.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.5/7.5 MB 102.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 78.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 8.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 140.1/140.1 kB 16.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.3/60.3 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 206.3/206.3 kB 21.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 80.4/80.4 kB 8.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.7/

In [2]:
import json, re, torch
import pandas as pd
import chromadb
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForCausalLM

print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE - switch runtime to T4 GPU")

# If you run out of GPU memory (or have no GPU), use the smaller model: "Qwen/Qwen2.5-1.5B-Instruct"
LLM_ID = "Qwen/Qwen2.5-3B-Instruct"

embedder = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
tok = AutoTokenizer.from_pretrained(LLM_ID)
model = AutoModelForCausalLM.from_pretrained(LLM_ID, torch_dtype="auto", device_map="auto")

def chat(prompt, max_new_tokens=300):
    msgs = [{"role": "user", "content": prompt}]
    text = tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True)
    inputs = tok(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(chat("Say hello in five words."))

GPU: Tesla T4


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

... (15 outputs left out)

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Hello, how can I help you?


## 1. Domain knowledge base
Swap this sample corpus for your own domain docs (dict of `source_name -> text`).

In [3]:
DOCS = {
 "chunking_guide": """Chunking splits documents into pieces small enough to embed and retrieve. Fixed-size chunks (e.g. 500 tokens) are simple but can cut ideas in half.

Overlap of 10-20% between chunks preserves context across boundaries. Semantic chunking splits on topic shifts and usually gives more coherent chunks at higher cost.""",
 "vector_stores": """A vector store indexes embeddings so that nearest-neighbour search is fast. Chroma is a lightweight open-source option that runs in-process, which makes it popular for prototypes.

Cosine similarity is the most common distance metric for text embeddings. Metadata filters let you restrict retrieval to a source, date, or tenant.""",
 "embeddings": """Embeddings map text to dense vectors so that semantically similar texts land close together. Task-specific embedding modes, such as separate query and document modes, usually improve retrieval quality.

Visualising embeddings with t-SNE or UMAP helps spot clusters and outliers in a corpus, though the plots are for intuition rather than measurement.""",
 "rag_evaluation": """The RAG triad evaluates three links: context relevance (did retrieval return useful chunks for the question), groundedness (is every claim in the answer supported by the retrieved context), and answer relevance (does the answer address the question asked).

Retrieval metrics such as MRR and nDCG need labelled relevant documents. LLM-as-a-judge scoring needs no labels but should be run deterministically and spot-checked by a human.""",
 "advanced_rag": """Query rewriting turns a vague or conversational question into a better search query before retrieval. Multi-query expansion retrieves for several paraphrases and merges the results.

Re-ranking takes the top 20-50 retrieved chunks and reorders them with a stronger cross-encoder or LLM, keeping only the best few for the prompt. This improves precision at the cost of latency.""",
 "failure_modes": """Hallucination in RAG usually appears when retrieval misses, and the model fills the gap from its own memory. Instructing the model to say it does not know when context is insufficient reduces this.

Citation errors occur when the model attributes a claim to the wrong chunk. Numbering chunks in the prompt and asking for bracketed citations like [1] makes verification straightforward.""",
}

def chunk(text, source):
    paras = [p.strip() for p in text.split("\n\n") if p.strip()]
    return [{"id": f"{source}-{i}", "text": p, "source": source} for i, p in enumerate(paras)]

CHUNKS = [c for s, t in DOCS.items() for c in chunk(t, s)]
print(len(CHUNKS), "chunks")

12 chunks


## 2. Embed and index in Chroma

In [4]:
def embed(texts):
    return embedder.encode(texts, normalize_embeddings=True).tolist()

chroma = chromadb.Client()
try: chroma.delete_collection("rag_kb")
except Exception: pass
kb = chroma.create_collection("rag_kb", metadata={"hnsw:space": "cosine"})
kb.add(
    ids=[c["id"] for c in CHUNKS],
    documents=[c["text"] for c in CHUNKS],
    metadatas=[{"source": c["source"]} for c in CHUNKS],
    embeddings=embed([c["text"] for c in CHUNKS]),
)

def retrieve(query, k=4):
    r = kb.query(query_embeddings=embed([query]), n_results=k)
    return [{"text": d, "source": m["source"]} for d, m in zip(r["documents"][0], r["metadatas"][0])]

## 3. The agent: retrieve, answer, cite

In [5]:
ANSWER_PROMPT = """You are a domain assistant. Answer ONLY using the numbered context below.
- Cite every claim with bracketed numbers like [1] or [2][3].
- If the context does not contain the answer, reply exactly: I don't know based on the available sources.
- Be concise (2-4 sentences).

CONTEXT:
<<CONTEXT>>

QUESTION: <<Q>>"""

def agent(question, k=4):
    ctx = retrieve(question, k)
    ctx_str = "\n\n".join(f"[{i+1}] (source: {c['source']}) {c['text']}" for i, c in enumerate(ctx))
    ans = chat(ANSWER_PROMPT.replace("<<CONTEXT>>", ctx_str).replace("<<Q>>", question), max_new_tokens=250)
    cited = sorted({int(n) for n in re.findall(r"\[(\d+)\]", ans) if 0 < int(n) <= len(ctx)})
    return {"question": question, "answer": ans, "context": ctx,
            "sources": [ctx[i-1]["source"] for i in cited]}

out = agent("Why is re-ranking used and what is its downside?")
print(out["answer"]); print("Sources:", out["sources"])

Re-ranking is used to take the top 20-50 retrieved chunks and reorder them with a stronger cross-encoder or LLM, keeping only the best few for the prompt. Its downside is that it improves precision at the cost of latency. [1]
Sources: ['advanced_rag']


## 4. RAG triad judges (LLM-as-a-judge, deterministic)
Small local models are less reliable at JSON, so each judge asks for one short answer (a number or YES/NO).

In [6]:
def score_0_10(prompt):
    out = chat(prompt, max_new_tokens=6)
    m = re.search(r"\d+", out)
    return min(int(m.group()), 10) / 10 if m else 0.0

def context_relevance(q, ctx):
    scores = []
    for c in ctx:
        p = ("Rate from 0 to 10 how useful the PASSAGE is for answering the QUESTION. "
             "Reply with ONLY one integer.\n\nQUESTION: " + q + "\n\nPASSAGE: " + c["text"])
        scores.append(score_0_10(p))
    return sum(scores) / len(scores)

DECLINE = re.compile(r"don't know|do not know|cannot answer|no information", re.I)

def groundedness(ans, ctx):
    ctx_str = "\n".join(f"[{i+1}] {c['text']}" for i, c in enumerate(ctx))
    sents = [re.sub(r"\[\d+\]", "", s).strip() for s in re.split(r"(?<=[.!?])\s+", ans)]
    sents = [s for s in sents if len(s) > 15 and not DECLINE.search(s)]
    if not sents:
        return 1.0   # a pure "I don't know" makes no unsupported claims
    ok = 0
    for s in sents:
        p = ("CONTEXT:\n" + ctx_str + "\n\nSTATEMENT: " + s +
             "\n\nIs the STATEMENT fully supported by the CONTEXT? Reply with ONLY YES or NO.")
        ok += chat(p, max_new_tokens=3).strip().upper().startswith("YES")
    return ok / len(sents)

def answer_relevance(q, ans):
    p = ("Rate from 0 to 10 how directly the ANSWER addresses the QUESTION (ignore whether it is correct). "
         "Reply with ONLY one integer.\n\nQUESTION: " + q + "\n\nANSWER: " + ans)
    return score_0_10(p)

def evaluate(question):
    o = agent(question)
    return {"question": question,
            "context_relevance": round(context_relevance(question, o["context"]), 2),
            "groundedness": round(groundedness(o["answer"], o["context"]), 2),
            "answer_relevance": round(answer_relevance(question, o["answer"]), 2),
            "sources": ", ".join(o["sources"]), "answer": o["answer"]}

## 5. Run the eval set
Includes an out-of-domain question to check the agent declines instead of hallucinating.

In [7]:
EVAL_QUESTIONS = [
    "What is the difference between fixed-size and semantic chunking?",
    "Which distance metric is most common for text embeddings?",
    "What are the three parts of the RAG triad?",
    "Why does hallucination happen in RAG and how can it be reduced?",
    "How does query rewriting help retrieval?",
    "What is the capital of Australia?",   # out-of-domain: should decline
]
results = pd.DataFrame([evaluate(q) for q in EVAL_QUESTIONS])
pd.set_option("display.max_colwidth", 80)
display(results[["question","context_relevance","groundedness","answer_relevance","sources"]])
print(results[["context_relevance","groundedness","answer_relevance"]].mean().round(2))

,question,context_relevance,groundedness,answer_relevance,sources
0,What is the difference between fixed-size and semantic chunking?,0.12,1.0,0.8,"chunking_guide, chunking_guide"
1,Which distance metric is most common for text embeddings?,0.07,1.0,1.0,vector_stores
2,What are the three parts of the RAG triad?,0.07,1.0,0.1,rag_evaluation
3,Why does hallucination happen in RAG and how can it be reduced?,0.10,1.0,0.1,failure_modes
4,How does query rewriting help retrieval?,0.10,1.0,0.1,advanced_rag
5,What is the capital of Australia?,0.00,1.0,0.0,


context_relevance    0.08
groundedness         1.00
answer_relevance     0.35
dtype: float64


## 6. Inspect answers
Read the actual answers next to the scores. Small-model judges are noisy, so spot-check the lowest rows by hand.

In [8]:
for _, r in results.iterrows():
    print("Q:", r["question"]); print("A:", r["answer"]); print("Sources:", r["sources"]); print("-"*60)

Q: What is the difference between fixed-size and semantic chunking?
A: Fixed-size chunking splits documents into pieces of a fixed size, such as 500 tokens, which can lead to cutting ideas in half. In contrast, semantic chunking splits on topic shifts to preserve context across boundaries and typically results in more coherent chunks at a higher computational cost. [1][2]
Sources: chunking_guide, chunking_guide
------------------------------------------------------------
Q: Which distance metric is most common for text embeddings?
A: Cosine similarity is the most common distance metric for text embeddings. [1]
Sources: vector_stores
------------------------------------------------------------
Q: What are the three parts of the RAG triad?
A: The RAG triad consists of three parts: context relevance, groundedness, and answer relevance. [1]
Sources: rag_evaluation
------------------------------------------------------------
Q: Why does hallucination happen in RAG and how can it be reduced?

## 7. Generate the SKILL.md
Send **`SKILL.md` + this notebook's agent code**. The test: can they run it on their own domain without asking you questions?

In [9]:
SKILL_MD = """---
name: cited-rag-agent
description: Answers domain questions from a document corpus with numbered citations, and scores each answer on the RAG triad (context relevance, groundedness, answer relevance). Runs fully free in Google Colab with open models. Use when you need a source-backed Q&A agent over your own documents.
---

# Cited RAG Agent

## When to use
- You have a set of documents and want answers that cite their sources.
- You want a repeatable quality score, not a gut check.

## Requirements
- Google Colab with a free T4 GPU runtime (no API keys)
- Models: all-MiniLM-L6-v2 (embeddings), Qwen2.5-3B-Instruct (agent and judge)

## Inputs
- `DOCS`: dict of `source_name -> text`
- `EVAL_QUESTIONS`: 5-10 realistic questions, including 1-2 the corpus cannot answer

## Steps
1. Chunk each document by paragraph (target 100-300 tokens; add 10-20% overlap for long prose).
2. Embed chunks with sentence-transformers and store in Chroma (cosine).
3. For each question: embed it, retrieve top-k (default 4).
4. Prompt the LLM with numbered context. Require bracketed citations like [1]. Require "I don't know" when context is insufficient.
5. Score deterministically (greedy decoding):
   - Context relevance: mean per-chunk usefulness, 0-10 scaled to 0-1
   - Groundedness: share of answer sentences judged fully supported by the context
   - Answer relevance: how directly the answer addresses the question
6. Read the lowest-scoring rows by hand.

## Output
- Answer text with [n] citations plus a list of source names
- A table of triad scores per question and the mean

## Interpreting scores
- Low context relevance -> fix chunking or k; add query rewriting or re-ranking
- Low groundedness -> tighten the prompt; the model is using outside knowledge
- Low answer relevance -> the question is ambiguous or retrieval missed; check the retrieved context

## Known limits
- Small-model judges are noisy; spot-check at least 10% of rows by hand.
- Out-of-domain questions should score groundedness 1.0 only if the agent declines.
- Paragraph chunking assumes reasonably structured source text.
- For stronger judging, swap `chat()` for a larger model.
"""
open("SKILL.md", "w").write(SKILL_MD)
from google.colab import files
files.download("SKILL.md")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>